In [ ]:
# Timeline scraper

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from typing import Dict, Any, List

def setup_driver():
    """Setup Chrome driver with options"""
    chrome_options = Options()
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--disable-blink-features=AutomationControlled')
    chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
    
    driver = webdriver.Chrome(options=chrome_options)
    return driver

def scrape_timeline(game_id: int) -> Dict[str, Any]:
    """
    Scrape timeline events and chart data from gol.gg timeline page
    
    Args:
        game_id: Game ID to scrape
    
    Returns:
        Dictionary containing events, gold_chart, and cs_chart data
    """
    driver = setup_driver()
    
    url = f"https://gol.gg/game/stats/{game_id}/page-timeline/"
    
    try:
        driver.get(url)
        
        # Wait for page to load
        wait = WebDriverWait(driver, 10)
        wait.until(EC.presence_of_element_located((By.TAG_NAME, "h1")))
        time.sleep(2)
        
        # Get page source
        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')
        
        # Extract events table
        events = parse_events_table(soup)

        # Click buttons to ensure charts are loaded (optional, but safe)
        try:
            gold_button = driver.find_element(By.ID, "goldgraph")
            gold_button.click()
            time.sleep(0.5)
        except:
            pass

        # Extract chart data from JavaScript
        gold_chart = extract_chart_data(driver, 'golddatas')

        try:
            cs_button = driver.find_element(By.ID, "csgraph")
            cs_button.click()
            time.sleep(0.5)
        except:
            pass

        cs_chart = extract_chart_data(driver, 'csdatas')
        
        return {
            'game_id': game_id,
            'events': events,
            'gold_chart': gold_chart,
            'cs_chart': cs_chart
        }
        
    except Exception as e:
        print(f"Error scraping timeline for game {game_id}: {e}")
        return None
        
    finally:
        driver.quit()

def parse_events_table(soup: BeautifulSoup) -> List[Dict[str, Any]]:
    """Parse the events table from timeline page"""
    events = []
    
    # Find the events table
    table = soup.find('table', class_='nostyle timeline trhover')
    
    if not table:
        print("Events table not found")
        return events
    
    rows = table.find('tbody').find_all('tr')
    
    for row in rows:
        # Skip header row
        if row.get('id') == 'lineheader':
            continue
            
        cols = row.find_all('td')
        
        if len(cols) < 5:
            continue
        
        event = {}
        
        # Time
        event['time'] = cols[0].text.strip()
        
        # Side (blue or red)
        side_img = cols[1].find('img')
        if side_img:
            src = side_img.get('src', '')
            if 'blueside' in src:
                event['side'] = 'BLUE'
            elif 'redside' in src:
                event['side'] = 'RED'
        
        # Player
        event['player'] = cols[2].text.strip()
        
        # Champion(s) involved
        champions = []
        champion_imgs = cols[3].find_all('img')
        for img in champion_imgs:
            champ_src = img.get('src', '')
            if 'champions_icon' in champ_src:
                champ_name = champ_src.split('/')[-1].replace('.png', '')
                champions.append(champ_name)
        event['champions'] = champions
        
        # Action type
        action_col = cols[4]
        action_img = action_col.find('img')
        if action_img:
            action_src = action_img.get('src', '')
            action_alt = action_img.get('alt', '')
            
            if 'kill-icon' in action_src:
                event['action'] = 'KILL'
            elif 'dragon' in action_src:
                if 'chemtech' in action_src:
                    event['action'] = 'CHEMTECH_DRAKE'
                elif 'hextech' in action_src:
                    event['action'] = 'HEXTECH_DRAKE'
                elif 'ocean' in action_src:
                    event['action'] = 'OCEAN_DRAKE'
                elif 'cloud' in action_src:
                    event['action'] = 'CLOUD_DRAKE'
                elif 'mountain' in action_src:
                    event['action'] = 'MOUNTAIN_DRAKE'
                elif 'infernal' in action_src:
                    event['action'] = 'INFERNAL_DRAKE'
                else:
                    event['action'] = 'DRAKE'
            elif 'herald' in action_src:
                event['action'] = 'RIFT_HERALD'
            elif 'nashor' in action_src or 'baron' in action_src:
                event['action'] = 'BARON'
            elif 'tower' in action_src:
                event['action'] = 'TOWER'
            elif 'inhib' in action_src:
                event['action'] = 'INHIBITOR'
            elif 'nexus' in action_src:
                event['action'] = 'NEXUS'
            elif action_alt:
                event['action'] = action_alt
        else:
            # Check for text-based actions (PLATE)
            action_text = action_col.text.strip()
            event['action'] = action_text if action_text else 'UNKNOWN'
        
        # Gold bounty if present
        gold_img = action_col.find('img', alt='Team Gold')
        if gold_img:
            gold_text = action_col.text.strip()
            gold_match = re.search(r'(\d+)', gold_text)
            if gold_match:
                event['gold_bounty'] = int(gold_match.group(1))
        
        # Target champion/structure
        if len(cols) > 6:
            target_col = cols[5]
            target_img = target_col.find('img')
            if target_img and 'champions_icon' in target_img.get('src', ''):
                target_src = target_img.get('src', '')
                event['target_champion'] = target_src.split('/')[-1].replace('.png', '')
            
            event['target'] = cols[6].text.strip()
        
        events.append(event)
    
    return events

def extract_chart_data(driver, var_name: str) -> Dict[str, Any]:
    """Extract chart data from JavaScript variable"""
    try:
        # Extract only the data we need, avoiding circular references
        script = f"""
        if (typeof {var_name} !== 'undefined') {{
            var result = {{
                labels: {var_name}.labels,
                datasets: []
            }};
            
            for (var i = 0; i < {var_name}.datasets.length; i++) {{
                result.datasets.push({{
                    label: {var_name}.datasets[i].label,
                    data: {var_name}.datasets[i].data
                }});
            }}
            
            return result;
        }}
        return null;
        """
        
        data = driver.execute_script(script)
        return data
        
    except Exception as e:
        print(f"Error extracting {var_name}: {e}")
        return None

def timeline_to_dataframe(timeline_data: Dict[str, Any]) -> pd.DataFrame:
    """Convert timeline data to a pandas DataFrame"""
    if not timeline_data or not timeline_data.get('events'):
        return pd.DataFrame()
    
    events = timeline_data['events']
    df = pd.DataFrame(events)
    df['game_id'] = timeline_data['game_id']
    
    # Reorder columns
    cols = ['game_id', 'time', 'side', 'player', 'action'] + [col for col in df.columns if col not in ['game_id', 'time', 'side', 'player', 'action']]
    df = df[[col for col in cols if col in df.columns]]
    
    return df

In [ ]:
# Loop through games

import pandas as pd
import json
from pathlib import Path
import time

games = pd.read_csv('data/games.csv')

# Paths
events_csv_path = Path("data/events.csv")
charts_json_path = Path("data/cs_gold.json")

# Ha létezik az events CSV, akkor beolvassuk, különben üres DF
if events_csv_path.exists():
    existing_events_df = pd.read_csv(events_csv_path)
else:
    existing_events_df = pd.DataFrame()

# Ha létezik a charts JSON, betöltjük, különben üres dict
if charts_json_path.exists():
    with open(charts_json_path, 'r', encoding='utf-8') as f:
        all_charts = json.load(f)
else:
    all_charts = {}

total_games = len(games)
start_time = time.time()  # Kezdési idő

for idx, game_id in enumerate(games['game_id'], start=1):
    if game_id in existing_events_df.game_id.unique():
        continue

    loop_start = time.time()
    
    print(f"\n[{idx}/{total_games}] Scraping game {game_id}...")
    timeline_data = scrape_timeline(game_id)
    
    if timeline_data:
        # ---- Events ----
        events_df = timeline_to_dataframe(timeline_data)
        events_df['game_id'] = game_id
        
        # Append a CSV-hez
        events_df.to_csv(events_csv_path, mode='a', header=not events_csv_path.exists(), index=False)
        print(f"✓ Appended events for game {game_id} to {events_csv_path}")
        
        # ---- Charts ----
        charts = {}
        if timeline_data.get('gold_chart'):
            charts['gold'] = timeline_data['gold_chart']
        if timeline_data.get('cs_chart'):
            charts['cs'] = timeline_data['cs_chart']
        
        if charts:
            all_charts[str(game_id)] = charts
            # Mentés JSON-be minden loop után
            with open(charts_json_path, 'w', encoding='utf-8') as f:
                json.dump(all_charts, f, ensure_ascii=False, indent=2)
            print(f"✓ Updated charts for game {game_id} in {charts_json_path}")
    
    loop_end = time.time()
    elapsed = loop_end - loop_start
    print(f"⏱ Time for this game: {elapsed:.2f}s")